# Session 17: Describing Data and Probability

Data Science with Python, Level 1, Stage 3: Statistics and ML Fundamentals

## Learning Objectives

By the end of this session you will be able to:

1. Tell a population from a sample.
2. Choose measures of centre and spread.
3. Use the normal curve and the 68-95-99.7 rule.
4. Calculate probabilities with three basic rules.
5. Estimate probabilities by simulation.
6. Find conditional probabilities in a table.

## Where We Are

Welcome to Stage 3. In Stage 2 you described data with words like *typical*, *spread out* and *unusual*. Statistics makes those words precise: which number to report as typical, how to measure spread, and how to tell whether one result is unusual compared with others. In the first half we describe the exam scores of 500 practice students.

Statistics also asks: how likely is a result **by chance**? In the second half you will calculate probabilities with a few simple rules, and estimate the ones that are too hard to calculate by **simulating** them thousands of times with NumPy. These ideas carry everything that follows: confidence intervals, tests and machine learning.

## Setup

Run this cell first. It loads the libraries and sets the address of the course datasets.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
exams = pd.read_csv(DATA_URL + "exam_scores.csv")

rng = np.random.default_rng(27)

## 1. Populations and Samples

## 2. Measuring the Centre

In [ ]:
math = exams["math"]

print("Mean:  ", round(math.mean(), 2))
print("Median:", math.median())
print("Mode:  ", math.mode()[0])

## 3. Measuring Spread

Two classes can have the same average but very different spreads. The **variance** is the average squared distance from the mean, and the **standard deviation** (std) is its square root, back in the original units. Roughly, the std is a typical distance from the mean.

pandas divides by `n - 1` when it computes the variance of a sample. This small correction makes the sample's variance a fair estimate of the population's.

In [ ]:
math = exams["math"]
print("Range:", math.max() - math.min())
print("Variance:", round(math.var(), 1))
print("Std:", round(math.std(), 1))
q1, q3 = math.quantile([0.25, 0.75])
print("IQR:", q3 - q1)

In [ ]:
class_a = pd.Series([58, 60, 61, 59, 62, 60, 60])
class_b = pd.Series([35, 80, 45, 75, 60, 90, 35])
for name, c in [("A", class_a), ("B", class_b)]:
    print(name, "mean", round(c.mean(), 1), "std", round(c.std(), 1))

| Data | Centre | Spread |
| --- | --- | --- |
| Roughly symmetric, no outliers | Mean | Standard deviation |
| Skewed or with outliers | Median | IQR |
| Categories | Mode (most common) | Counts or shares |

## 4. The Normal Distribution

Many measurements, such as heights, measurement errors and exam scores, have a **bell-shaped** distribution: most values near the middle, fewer and fewer further away, symmetric on both sides. The ideal version is the **normal distribution**, described by just two numbers: its mean and its standard deviation.

In [ ]:
mu, sigma = math.mean(), math.std()
x = np.linspace(10, 100, 200)

sns.histplot(math, bins=25, stat="density", alpha=0.5)
plt.plot(x, stats.norm.pdf(x, mu, sigma), color="tab:red", linewidth=2)
plt.title("Math scores and a normal curve with the same mean and std")
plt.show()

In [ ]:
for k in [1, 2, 3]:
    inside = math.between(mu - k * sigma, mu + k * sigma).mean()
    print(f"Within {k} std: {inside:.1%}")

## 5. z-Scores and Percentiles

In [ ]:
exams["math_z"] = (exams["math"] - mu) / sigma

student = exams.loc[0, ["math", "math_z"]]
print(student.round(2))
print("Percentile:", round(stats.percentileofscore(exams["math"], student["math"]), 1))

The second half uses random numbers. Resetting the generator here makes your results match the slides.

In [ ]:
rng = np.random.default_rng(27)

## 6. What Is a Probability?

In [ ]:
rolls = rng.integers(1, 7, size=12)
print(rolls)
print("Share of sixes:", (rolls == 6).mean())

many = rng.integers(1, 7, size=100_000)
print("Share of sixes in 100,000 rolls:", (many == 6).mean())
print("Exact:", round(1 / 6, 5))

## 7. Rules of Probability

| Rule | Formula | Example |
| --- | --- | --- |
| Complement (not) | P(not A) = 1 - P(A) | P(no six) = 1 - 1/6 = 5/6 |
| And, independent events | P(A and B) = P(A) x P(B) | Two sixes in two rolls: 1/36 |
| Or, events that cannot both happen | P(A or B) = P(A) + P(B) | A 1 or a 2: 2/6 |
| At least one | 1 - P(none) | At least one six in 4 rolls: 1 - (5/6)^4 |

In [ ]:
exact = 1 - (5 / 6) ** 4

games = rng.integers(1, 7, size=(100_000, 4))
simulated = (games == 6).any(axis=1).mean()

print("Exact:    ", round(exact, 4))
print("Simulated:", round(simulated, 4))

## 8. Simulation

In [ ]:
def shared_birthday(people, trials=20_000):
    days = rng.integers(1, 366, size=(trials, people))
    shared = [len(set(row)) < people for row in days]
    return np.mean(shared)

for people in [10, 23, 30, 50]:
    print(people, "people:", round(shared_birthday(people), 3))

## 9. The Law of Large Numbers

In [ ]:
flips = rng.integers(0, 2, size=5000)
running_share = flips.cumsum() / np.arange(1, 5001)

plt.plot(running_share)
plt.axhline(0.5, color="tab:red", linestyle="--")
plt.title("Share of heads as the number of flips grows")
plt.xlabel("Number of flips")
plt.ylabel("Share of heads")
plt.show()

## 10. Conditional Probability

A **conditional probability** P(A given B) is the probability of A among only the cases where B happened. With data, it is a share inside a group, which is exactly what `groupby` and `crosstab` calculate. Here we use a practice dataset of 2,000 customers of a fictional phone company, some of whom cancelled their contract, or **churned**.

In [ ]:
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
churn["left"] = churn["churn"] == "Yes"

print("P(left):", round(churn["left"].mean(), 3))
print(churn.groupby("contract")["left"].mean().round(3))

## 11. The Binomial Distribution

When the same yes or no experiment is repeated `n` times with the same probability `p`, the number of successes follows a **binomial distribution**. Examples: the number of sixes in 10 rolls, the number of customers out of 20 who buy, the number of faulty items in a box of 50.

In [ ]:
n, p = 20, 0.3
k = np.arange(0, n + 1)
probs = stats.binom.pmf(k, n, p)

plt.bar(k, probs)
plt.title("Customers who buy, out of 20, when p = 0.3")
plt.xlabel("Number who buy")
plt.ylabel("Probability")
plt.show()
print("P(exactly 6):", round(stats.binom.pmf(6, n, p), 3))
print("P(10 or more):", round(1 - stats.binom.cdf(9, n, p), 3))

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| Hidden spread | Reporting an average without a spread | Always give the std or the IQR next to the mean or median. |
| Skewed summary | Using the mean and std for skewed data | Use the median and IQR, or transform the data first. |
| Gambler's fallacy | Thinking a coin is due for heads after five tails | Independent flips have no memory. |
| Confused direction | Treating P(A given B) as P(B given A) | Say carefully which group the share is taken in. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Describe Physics

Report the mean, median, std and IQR of the physics scores. Which pair of measures would you report, and why?

In [ ]:
# Write your code below

### In-Class Exercise 2: Top Ten Percent

What math score is needed to be in the top 10 percent? Use `quantile(0.9)`. How many students reach it?

In [ ]:
# Write your code below

### In-Class Exercise 3: Two Dice

Simulate 100,000 rolls of two dice and estimate the probability that the total is 7. Compare with the exact value 6/36.

In [ ]:
# Write your code below

### In-Class Exercise 4: Exam Guessing

A quiz has 10 questions with 4 options each. A student guesses every answer. Use `stats.binom` to find the chance of passing with 5 or more correct.

In [ ]:
# Write your code below

## Summary

- A sample is a part of the population; statistics describe samples.
- Report the mean and std for symmetric data, the median and IQR for skewed data.
- In a normal curve, about 68, 95 and 99.7 percent lie within 1, 2 and 3 std.
- A z-score says how many std a value is from the mean.
- Complement, addition and multiplication rules combine probabilities.
- Simulation estimates any probability; conditional probabilities come from tables.

## What's Next

A sample mean changes from sample to sample. Next session you will measure how much with confidence intervals, and decide whether a difference is real or just chance with hypothesis tests.